In [ ]:
import pandas as pd
import numpy as np

# ===== PATHS =====
naps_csv = r"C:\Users\fkamdar\Desktop\repos\pd_nonmotor\stimuli\all_NAPs_ratings.csv"
old_csv = r"C:\Users\fkamdar\Desktop\repos\pd_nonmotor\stimuli\blocks\pndm01_block_112.csv"
iaps_csv = r"C:\Users\fkamdar\Desktop\repos\pd_nonmotor\stimuli\all_IAPs_ratings.csv"
out_dir = r"C:\Users\fkamdar\Desktop\repos\pd_nonmotor\stimuli\blocks"

SEED = 42
np.random.seed(SEED)

sessions = 5
TARGET_TOTAL = 112

# ===== LOAD DATA =====
df_naps = pd.read_csv(naps_csv)
old_df = pd.read_csv(old_csv)

# remove session1 images
df_naps = df_naps[~df_naps['ID'].isin(old_df['ID'])]

df_iaps = pd.read_csv(iaps_csv)
df_iaps["Valence"] = 10 - df_iaps["Valence"]

# keep only extreme bins for IAPS
df_iaps = df_iaps[(df_iaps['Valence'] <= 2) | (df_iaps['Valence'] >= 8)]

# merge
df = pd.concat([df_naps, df_iaps], ignore_index=True)
df = df[['ID', 'Valence', 'Category']]

# ===== BINNING =====
bins = [1,2,3,4,5,6,7,8,9]
bin_labels = [f"{bins[i]}-{bins[i+1]}" for i in range(len(bins)-1)]

df["val_bin"] = pd.cut(df["Valence"], bins=bins, labels=bin_labels, include_lowest=True)

counts = df["val_bin"].value_counts().reindex(bin_labels)
print("Initial counts:\n", counts)

# ===== PER-BIN TARGETS =====
bin_counts = counts.to_dict()

per_session_targets = {}

for b in bin_labels:
    per_session_targets[b] = bin_counts[b] // sessions

# adjust to reach 112
adjustable_bins = [b for b in bin_labels if b not in ["1-2", "8-9"]]

current_total = sum(per_session_targets.values())
i = 0

while current_total < TARGET_TOTAL:
    b = adjustable_bins[i % len(adjustable_bins)]
    per_session_targets[b] += 1
    current_total += 1
    i += 1

print("\nPer-session targets:")
print(per_session_targets)
print("Total per session:", sum(per_session_targets.values()))

# ===== PREPARE BIN POOLS =====
rng = np.random.default_rng(SEED)

bin_pools = {}
for b in bin_labels:
    pool = df[df["val_bin"] == b].copy()
    pool = pool.sample(frac=1, random_state=SEED).reset_index(drop=True)
    bin_pools[b] = pool

# ===== GENERATE SESSIONS =====
sessions_data = []

for s in range(sessions):

    selected = []

    for b in bin_labels:
        n_take = per_session_targets[b]

        pool = bin_pools[b]

        if len(pool) < n_take:
            raise ValueError(f"Bin {b} ran out of images in session {s+1}")

        take = pool.iloc[:n_take]
        selected.append(take)

        # remove used (no reuse)
        bin_pools[b] = pool.iloc[n_take:].reset_index(drop=True)

    block = pd.concat(selected).reset_index(drop=True)

    # shuffle
    block = block.sample(frac=1, random_state=SEED + s).reset_index(drop=True)

    # ===== ASSIGN CONDITION (PER-BIN BALANCED) =====

    block["condition"] = None


    # ===== FINAL FORMAT =====
    block["trial"] = np.arange(1, len(block)+1)
    block["filename"] = block["ID"] + ".jpg"

    out_csv = rf"{out_dir}\session_{s+2}_block_112.csv"

    block[["trial","ID","filename","Category","Valence","val_bin","condition"]] \
        .to_csv(out_csv, index=False)

    print(f"\nSaved: {out_csv}")
    print(block.groupby(["val_bin","condition"]).size())

    sessions_data.append(block)

print("\nAll sessions generated successfully.")